# Deploy AI Healthcare Assistant to GCP Cloud Run
## Run each cell in order

In [2]:
# Cell 1: Set your GCP Project ID here
PROJECT_ID = 'healthcare-ai-platform-498222'  # Replace this!
REGION     = 'us-central1'
SERVICE    = 'healthcare-ai'
IMAGE      = f'gcr.io/{PROJECT_ID}/{SERVICE}'

print('Project ID:', PROJECT_ID)
print('Image URI: ', IMAGE)

Project ID: healthcare-ai-platform-498222
Image URI:  gcr.io/healthcare-ai-platform-498222/healthcare-ai


In [4]:
# Cell 2: Write .dockerignore to keep image small
dockerignore = """__pycache__
*.pyc
*.pyo
.git
.ipynb_checkpoints
*.ipynb
healthcare_ai/uploads/images
"""
with open('.dockerignore', 'w', encoding='utf-8') as f:
    f.write(dockerignore)
print('Created .dockerignore')

Created .dockerignore


In [6]:
# Cell 3: Write Dockerfile
dockerfile = """FROM python:3.10-slim

RUN apt-get update && apt-get install -y \\
    libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 curl \\
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY healthcare_ai/ ./healthcare_ai/

EXPOSE 8080

CMD ["uvicorn", "healthcare_ai.api.main:app", "--host", "0.0.0.0", "--port", "8080"]
"""
with open('Dockerfile', 'w', encoding='utf-8') as f:
    f.write(dockerfile)
print('Created Dockerfile')

Created Dockerfile


In [8]:
# Cell 4: Update main.py port for Cloud Run (uses 8080)
import re
with open('healthcare_ai/api/main.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Cloud Run uses PORT env variable
old = 'if __name__ == "__main__":'
new = """if __name__ == "__main__":
    import os as _os
    port = int(_os.environ.get("PORT", 8080))
    import uvicorn as _uvicorn
    _uvicorn.run("main:app", host="0.0.0.0", port=port)
"""
# Also update DB path to use absolute path
content = content.replace(
    'DB_PATH   = "healthcare_ai/data/healthcare.db"',
    'DB_PATH   = "/app/healthcare_ai/data/healthcare.db"'
)
content = content.replace(
    'MODEL_DIR = "healthcare_ai/models"',
    'MODEL_DIR = "/app/healthcare_ai/models"'
)
content = content.replace(
    'IMG_DIR   = "healthcare_ai/uploads/images"',
    'IMG_DIR   = "/app/healthcare_ai/uploads/images"'
)
with open('healthcare_ai/api/main.py', 'w', encoding='utf-8') as f:
    f.write(content)
print('main.py updated for Cloud Run')

main.py updated for Cloud Run


In [10]:
# Cell 5: Generate deploy script
import os
script = f"""gcloud auth login
gcloud config set project {PROJECT_ID}
gcloud services enable cloudbuild.googleapis.com run.googleapis.com containerregistry.googleapis.com
gcloud builds submit --tag {IMAGE} .
gcloud run deploy {SERVICE} --image {IMAGE} --platform managed --region {REGION} --allow-unauthenticated --memory 4Gi --cpu 2 --timeout 300 --port 8080
"""
with open('deploy.bat', 'w', encoding='utf-8') as f:
    f.write(script)
print('Created deploy.bat')
print()
print('NEXT STEPS:')
print('1. Open Command Prompt')
print(f'2. cd "C:\\Users\\kssud\\AI Healthcare Assistant platform"')
print('3. deploy.bat')
print('4. Sign in to Google when browser opens')
print('5. Wait 5-10 minutes for build + deploy')
print('6. Get your public URL!')

Created deploy.bat

NEXT STEPS:
1. Open Command Prompt
2. cd "C:\Users\kssud\AI Healthcare Assistant platform"
3. deploy.bat
4. Sign in to Google when browser opens
5. Wait 5-10 minutes for build + deploy
6. Get your public URL!


In [ ]:
# Cell 6: After deploying - get your live URL
import subprocess
result = subprocess.run(
    ['gcloud', 'run', 'services', 'describe', SERVICE,
     '--region', REGION, '--format', 'value(status.url)'],
    capture_output=True, text=True
)
url = result.stdout.strip()
if url:
    print('Your live app URL:')
    print(f'  Frontend: {url}/app')
    print(f'  API Docs: {url}/docs')
    print(f'  Dashboard: {url}/dashboard')
else:
    print('Run deploy.bat first, then run this cell')